In [1]:
import os
import numpy as np
import pandas as pd
import boto3
import time
import helper
import sys
import matplotlib.pyplot as plt
cwd = os.getcwd()
print(cwd)

/home/sagemaker-user/CAPE_PERFORMANCE


#### Load the data

In [25]:

versions = ['v4', 'v5']
splits = ['train', 'val']
srcs = ['model', 'cape', 'base']
model_prefix = 'Projects/AdHoc/InternProjects/2025/2025FallCapeV5_vs_V4_performance_analysis/all_matched_response_OK/'

version = versions[0]
SPLIT_TRAIN = 1

bucket_name = 'pr-home-datascience'
s3 = boto3.client('s3')

file_name = f'filtered_{srcs[0]}_{splits[1]}_{version}.csv'
model_val = helper.read_s3(bucket_name, model_prefix, file_name)

file_name = f'filtered_{srcs[0]}_{splits[0]}_{version}.csv'
model_train = helper.read_s3(bucket_name, model_prefix, file_name)


file_name = f'filtered_{srcs[1]}_{splits[1]}_{version}.csv'
cape_val = helper.read_s3(bucket_name, model_prefix, file_name)

file_name = f'filtered_{srcs[1]}_{splits[0]}_{version}.csv'
cape_train = helper.read_s3(bucket_name, model_prefix, file_name)

file_name = f'filtered_{srcs[2]}_{splits[1]}_{version}.csv'
base_val = helper.read_s3(bucket_name, model_prefix, file_name)

file_name = f'filtered_{srcs[2]}_{splits[0]}_{version}.csv'
base_train = helper.read_s3(bucket_name, model_prefix, file_name)

Reading from the file  Projects/AdHoc/InternProjects/2025/2025FallCapeV5_vs_V4_performance_analysis/all_matched_response_OK/filtered_model_val_v4.csv
Load file successfully, file length is  248953
Now the total rows are  248953
Reading from the file  Projects/AdHoc/InternProjects/2025/2025FallCapeV5_vs_V4_performance_analysis/all_matched_response_OK/filtered_model_train_v4.csv
Load file successfully, file length is  584791
Now the total rows are  584791
Reading from the file  Projects/AdHoc/InternProjects/2025/2025FallCapeV5_vs_V4_performance_analysis/all_matched_response_OK/filtered_cape_val_v4.csv
Load file successfully, file length is  248953
Now the total rows are  248953
Reading from the file  Projects/AdHoc/InternProjects/2025/2025FallCapeV5_vs_V4_performance_analysis/all_matched_response_OK/filtered_cape_train_v4.csv
Load file successfully, file length is  584791
Now the total rows are  584791
Reading from the file  Projects/AdHoc/InternProjects/2025/2025FallCapeV5_vs_V4_perform

In [26]:
print(f'Cape train has length of {len(cape_train)} with variables of {len(cape_train.columns)}')
print(f'Cape val has length of {len(cape_val)} with variables of {len(cape_val.columns)}')
print('')
print(f'Model train has length of {len(model_train)} with variables of {len(model_train.columns)}')
print(f'Model val has length of {len(model_val)} with variables of {len(model_val.columns)}')
print('')
print(f'Baseline train has length of {len(base_train)} with variables of {len(base_train.columns)}')
print(f'Baseline val has length of {len(base_val)} with variables of {len(base_val.columns)}')

Cape train has length of 584791 with variables of 103
Cape val has length of 248953 with variables of 103

Model train has length of 584791 with variables of 15
Model val has length of 248953 with variables of 15

Baseline train has length of 584791 with variables of 119
Baseline val has length of 248953 with variables of 119


In [27]:
# sanity check
print(base_train[['pol_num', 'year']][200:210])
print(model_train[['pol_num', 'year', 'cur_term_eff_dt']][200:210])
print(cape_train[['pol_num', 'cape_run_dt']][200:210])

            pol_num  year
200  BHD00001001777  2015
201  BHD00001001777  2016
202  BHD00001001777  2017
203  BHD00001001777  2018
204  BHD00001001800  2014
205  BHD00001001800  2015
206  BHD00001001800  2016
207  BHD00001001800  2017
208  BHD00001001800  2018
209  BHD00001001800  2019
            pol_num  year cur_term_eff_dt
200  BHD00001001777  2015      2015-02-02
201  BHD00001001777  2016      2016-02-02
202  BHD00001001777  2017      2017-02-02
203  BHD00001001777  2018      2018-02-02
204  BHD00001001800  2014      2014-03-01
205  BHD00001001800  2015      2015-03-01
206  BHD00001001800  2016      2016-03-01
207  BHD00001001800  2017      2017-03-01
208  BHD00001001800  2018      2018-03-01
209  BHD00001001800  2019      2019-03-01
            pol_num cape_run_dt
200  BHD00001001777  2015-01-02
201  BHD00001001777  2016-01-02
202  BHD00001001777  2017-01-02
203  BHD00001001777  2018-01-02
204  BHD00001001800  2014-02-01
205  BHD00001001800  2015-02-01
206  BHD00001001800  2016-02

In [28]:
model_train['ncat'].describe()

count    5.847910e+05
mean     6.095588e+02
std      1.708489e+04
min      0.000000e+00
25%      0.000000e+00
50%      0.000000e+00
75%      0.000000e+00
max      6.864000e+06
Name: ncat, dtype: float64

#### Adjusted Ncat

Add the accumulated inflation factor the the Ncat.

In [29]:
model_train = helper.add_cumulative_inflation(
    model_train, year_col = "year",
    loss_col= "ncat",
    ee_col= "ee",
    compute_pp =True,
)

model_val = helper.add_cumulative_inflation(
    model_val, year_col = "year",
    loss_col= "ncat",
    ee_col= "ee",
    compute_pp = True,
)


The inflation adjusted ncat is added into the dataframe as ncat_infl_adj.
The inflation adjusted pp is added into the dataframe as pp_infl_adj.
The inflation adjusted ncat is added into the dataframe as ncat_infl_adj.
The inflation adjusted pp is added into the dataframe as pp_infl_adj.


In [30]:
helper.keywords_in_var('ncat', model_train)

keywords are not in iterable
There are  3  variables contains ncat.
ncat 
ncat_cnt 
ncat_infl_adj


In [31]:
# print(model_val['ee'].nlargest(10))
model_train.loc[model_train['ee']>1,'ee']=1
model_val.loc[model_val['ee']>1,'ee']=1

model_train.loc[model_train['ncat']>750000,'ncat']=750000
# print(model_train['ncat'].nlargest(20))
model_val.loc[model_val['ncat']>750000,'ncat']=750000

#### Prepare dataset

Only keep the baseline annulized loss for future purpose.

In [32]:
base_train_need = base_train['p0_alxc_wo_cape_pred']
base_val_need = base_val['p0_alxc_wo_cape_pred']

In [33]:
model_train_all = pd.concat(
    [cape_train, model_train, base_train_need],
    axis=1
)

# keep only the one pol_num and state
model_train_all = model_train_all.loc[:, ~model_train_all.columns.duplicated()]

model_val_all = pd.concat(
    [cape_val, model_val, base_val_need],
    axis=1
)

# keep only the one pol_num and state
model_val_all = model_val_all.loc[:, ~model_val_all.columns.duplicated()]

#### Splity the dataset

Split the train into new_train and new_valid. The original valid becomes holdout.

In [34]:
from sklearn.model_selection import train_test_split



if SPLIT_TRAIN == 1:
    strata = (
            model_train_all[['year', 'state', 'co_cd']]
            .astype(str)
            .agg('_'.join, axis=1)
        )
    model_train_split, model_test_split = train_test_split(
        model_train_all,
        test_size=0.3,
        random_state=42,
        shuffle=True,
        stratify=strata,
    )

elif SPLIT_TRAIN == 0:
    strata = (
            model_val_all[['year', 'state', 'co_cd']]
            .astype(str)
            .agg('_'.join, axis=1)
        )
    model_val_split, model_test_split = train_test_split(
        model_val_all,
        test_size=0.33,
        random_state=42,
        shuffle=True,
        stratify=strata,
    )

In [35]:
if SPLIT_TRAIN == 1:
    # output_path = f'./{version}_datasets/'
    output_path = f's3://pr-home-datascience/Projects/AdHoc/InternProjects/2025/2025FallCapeV5_vs_V4_performance_analysis/Train_val_holdout_final/{version}/'
    model_val_split = model_val_all
elif SPLIT_TRAIN ==0:
    output_path = f'./{version}_datasets_val_split/'
    model_train_split = model_train_all


model_train_split.to_csv(output_path+'train.csv')
model_test_split.to_csv(output_path+'valid.csv')
model_val_split.to_csv(output_path+'holdout.csv')


#### Sanity check and save data

In [13]:
from openpyxl import Workbook, load_workbook
!pip install XlsxWriter
import xlsxwriter

def eda_train_val_test(
    combined_train: pd.DataFrame,
    combined_val: pd.DataFrame,
    combined_test: pd.DataFrame,
    dist_cols: list[str],   # e.g. ["year", "state", "co_cd"]
    output_file: str = "eda_summary.xlsx",
):
    # work on copies so we don't pollute original dfs
    train_df = combined_train.copy()
    test_df  = combined_test.copy()
    val_df   = combined_val.copy()

    with pd.ExcelWriter(output_file, engine="xlsxwriter") as writer:
        for dist_col in dist_cols:
            # --- 1️⃣ Optional special handling for co_cd ---
            if dist_col == "co_cd":
                keywords = "ALN_"

                mask_train = train_df[dist_col].str.startswith(keywords, na=False)
                mask_test  = test_df[dist_col].str.startswith(keywords, na=False)
                mask_val   = val_df[dist_col].str.startswith(keywords, na=False)

                # two-bucket summary: "home (aln_)" vs "other"
                train_df["legacy_bucket"] = mask_train.map({True: "home (aln_)", False: "other"})
                test_df["legacy_bucket"]  = mask_test.map({True: "home (aln_)", False: "other"})
                val_df["legacy_bucket"]   = mask_val.map({True: "home (aln_)", False: "other"})

                group_key = "legacy_bucket"   # or dist_col if you want exact co_cd
            else:
                group_key = dist_col

            # --- 2️⃣ Counts by group_key ---
            train_cnt = (
                train_df.groupby(group_key, dropna=False)
                        .size()
                        .rename("Training_Count")
            )
            test_cnt = (
                test_df.groupby(group_key, dropna=False)
                       .size()
                       .rename("Test_Count")
            )
            val_cnt = (
                val_df.groupby(group_key, dropna=False)
                      .size()
                      .rename("Validation_Count")
            )

            dist_df = (
                pd.concat([train_cnt,  val_cnt, test_cnt,], axis=1)
                  .fillna(0)
                  .astype(int)
                  .sort_index()
            )

            # --- 3️⃣ Percentages ---
            t_sum = dist_df["Training_Count"].sum()
            
            v_sum = dist_df["Validation_Count"].sum()
            s_sum = dist_df["Test_Count"].sum()

            dist_df["Training_Pct"]   = dist_df["Training_Count"]   / t_sum if t_sum > 0 else 0.0
            dist_df["Validation_Pct"] = dist_df["Validation_Count"] / v_sum if v_sum > 0 else 0.0
            dist_df["Test_Pct"]       = dist_df["Test_Count"]       / s_sum if s_sum > 0 else 0.0

            dist_df.index.name = group_key

            # --- 4️⃣ Total row ---
            total_row = pd.DataFrame({
                dist_df.index.name: ["Total"],
                "Training_Count":   [t_sum],
                "Validation_Count": [v_sum],
                "Test_Count":       [s_sum],
                "Training_Pct":     [1.0 if t_sum > 0 else 0.0],
                "Validation_Pct":   [1.0 if v_sum > 0 else 0.0],
                "Test_Pct":         [1.0 if s_sum > 0 else 0.0],
            })

            final_df = pd.concat([dist_df.reset_index(), total_row], ignore_index=True)

            # --- 5️⃣ Write to Excel ---
            sheet_name = f"Dist_by_{group_key}"
            final_df.to_excel(writer, sheet_name=sheet_name, index=False)

            workbook  = writer.book
            worksheet = writer.sheets[sheet_name]
            pct_fmt   = workbook.add_format({'num_format': '0.0%'})
            # columns: A=0, B=1, C=2, D=3, E=4, F=5, G=6
            # Training_Pct, Test_Pct, Validation_Pct should be cols F,G,H
            # assuming columns are:
            # [group_key, Training_Count, Test_Count, Validation_Count,
            #  Training_Pct, Test_Pct, Validation_Pct]
            worksheet.set_column('E:G', 15, pct_fmt)

    print(f"EDA summary written to: {output_file}")
    print(f"Total training: {t_sum:,}, test: {s_sum:,}, validation: {v_sum:,}")

  Using cached xlsxwriter-3.2.9-py3-none-any.whl.metadata (2.7 kB)
Using cached xlsxwriter-3.2.9-py3-none-any.whl (175 kB)


In [14]:
output_file = f'{output_path}sanity_check_tvt_{version}.xlsx'

eda_train_val_test(
    model_train_split,
    model_val_split,
    model_test_split,
    dist_cols=["year", "state", "co_cd"],
    output_file=output_file,
)

EDA summary written to: ./v5_datasets/sanity_check_tvt_v5.xlsx
Total training: 409,353, test: 175,438, validation: 248,953


In [15]:
def calc_overall_summary_adj(df_train, df_val, df_test):
    """
    Summarize overall EE / NCAT_adj / EPRM / ncat_cnt for
    train, validation, and test datasets.
    """
    def summarize(df, name):
        df = df.copy()

        # ensure numeric
        for col in ["ee", "ncat_infl_adj", "eprm", "ncat_cnt"]:
            if col in df.columns:
                df[col] = pd.to_numeric(df[col], errors="coerce")

        total_pol  = df["pol_num"].shape[0]
        sum_ee     = df["ee"].sum(skipna=True)
        sum_ncat   = df["ncat_infl_adj"].sum(skipna=True)
        sum_eprm   = df["eprm"].sum(skipna=True)
        sum_claims = df["ncat_cnt"].sum(skipna=True) if "ncat_cnt" in df.columns else float("nan")

        # core metrics
        avg_ee    = sum_ee / total_pol if total_pol > 0 else float("nan")
        pure_prem = sum_ncat / sum_ee  if sum_ee > 0 else float("nan")
        freq      = sum_claims / sum_ee if (sum_ee > 0 and pd.notna(sum_claims)) else float("nan")
        severity  = sum_ncat / sum_claims if (pd.notna(sum_claims) and sum_claims > 0) else float("nan")
        LR        = sum_ncat / sum_eprm if sum_eprm > 0 else float("nan")

        return pd.Series({
            "Dataset": name,
            "Total_Policies": total_pol,
            "Sum_EE": sum_ee,
            "Sum_NCAT_adj": sum_ncat,
            "Sum_EPRM": sum_eprm,
            "Overall PP_adj (NCAT_adj/EE)": pure_prem,                     # sum(NCAT_adj)/sum(EE)
            "Sum_Claims (ncat_cnt)": sum_claims,
            "Avg_EE_per_Policy (ee/#)": avg_ee,
            "Overall Claim_Frequency_adj (ncat_cnt/ee)": freq,             # sum(ncat_cnt)/sum(EE)
            "Overall Avg_Severity_adj (ncat_adj/ncat_cnt)": severity,      # sum(NCAT_adj)/sum(ncat_cnt)
            "Loss_Ratio_adj (NCAT_adj/EPRM)": LR
        })

    summary_df = pd.concat([
        summarize(df_train, "Training"),
        summarize(df_val,   "Validation"),
        summarize(df_test,  "Test"),
    ], axis=1).T.reset_index(drop=True)

    # column names updated to match *_adj labels above
    round_map = {
        "Sum_EE": 2,
        "Sum_NCAT_adj": 2,
        "Sum_EPRM": 2,
        "Overall PP_adj (NCAT_adj/EE)": 2,
        "Sum_Claims (ncat_cnt)": 0,
        "Avg_EE_per_Policy (ee/#)": 3,
        "Overall Claim_Frequency_adj (ncat_cnt/ee)": 6,
        "Overall Avg_Severity_adj (ncat_adj/ncat_cnt)": 2,
        "Loss_Ratio_adj (NCAT_adj/EPRM)": 6,
    }

    # ensure numeric for rounding targets
    for col, ndigits in round_map.items():
        if col in summary_df.columns:
            summary_df[col] = pd.to_numeric(summary_df[col], errors="coerce")

    summary_df = summary_df.round(round_map)

    return summary_df

In [16]:
output_file = f'{output_path}sanity_check_tvt_{version}.xlsx'

summary_all = calc_overall_summary_adj(
    model_train_split,
    model_val_split,
    model_test_split,
)
with pd.ExcelWriter(output_file, engine="openpyxl", mode="a", if_sheet_exists="new") as writer:
    summary_all.to_excel(writer, sheet_name="Overall_Summary_adj", index=False)

helper.modify_excel(output_file)

The Excel is successfully modified
